In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import Swarm
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.conditions import TextMentionTermination,HandoffTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


model_client=OpenAIChatCompletionClient(model='gpt-4o',parallel_tool_calls=False)

In [2]:
### define tool

def refund_flight(flight_pnr:str)->str:
    return f"Flight {flight_pnr} refunded"

In [3]:
### define agent

travel_agent=AssistantAgent(
    name='travel_planning_agent',
    model_client=model_client,
    description="A agent that assist the end user related to any travel,flight,booking queries",
    handoffs=['flights_refunder_agent','user'],
    system_message="""You are a travel agent. You task is to help the user any travel related 
    issue only.If user asked different query then response as you are not aware the issue 
    and mention TERMINATE to exit the chat.
    If the issue collaborating with other tools if required.
    The flights_refunder_agent is in charge of refunding flights.
    If you need information from the user, you must first send your message, 
    then you can handoff to the user.
    Use TERMINATE when the travel planning is completed. """
)


flights_refund=AssistantAgent(
    name="flights_refunder_agent",
    model_client=model_client,
    description="A Agent that works for refunding booking amount to the user.",
    tools=[refund_flight],
    handoffs=['travel_planning_agent','user'],
    system_message="""
    You are an agent specialized in refunding flights.
    You only need flight PNR to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.
"""

)

In [4]:
termination=HandoffTermination('user') | TextMentionTermination('TERMINATE')

team=Swarm(participants=[travel_agent,flights_refund],
           termination_condition=termination,max_turns=15)

In [5]:
task="I need refund of my flight booking."

async def main(task:str)->None:
    task_result=await Console(team.run_stream(task=task))
    last_message=task_result.messages[-1]

    while isinstance(last_message,HandoffMessage) and last_message.target=='user':
        user_input=input("User: ")
        task_result=await Console(team.run_stream(
            task=HandoffMessage(source='user',content=user_input,target=last_message.source)))
        last_message=task_result.messages[-1]

        

In [6]:
await main(task=task)

---------- TextMessage (user) ----------
I need refund of my flight booking.
---------- ToolCallRequestEvent (travel_planning_agent) ----------
[FunctionCall(id='call_kHjn5bkxllzwyKWXG8uHvSvw', arguments='{}', name='transfer_to_flights_refunder_agent')]
---------- ToolCallExecutionEvent (travel_planning_agent) ----------
[FunctionExecutionResult(content='Transferred to flights_refunder_agent, adopting the role of flights_refunder_agent immediately.', name='transfer_to_flights_refunder_agent', call_id='call_kHjn5bkxllzwyKWXG8uHvSvw', is_error=False)]
---------- HandoffMessage (travel_planning_agent) ----------
Transferred to flights_refunder_agent, adopting the role of flights_refunder_agent immediately.
---------- TextMessage (flights_refunder_agent) ----------
To help you with your refund, could you please provide me with your flight PNR number?
---------- ToolCallRequestEvent (flights_refunder_agent) ----------
[FunctionCall(id='call_cXG1gsd395EgAt2uVgkeCeot', arguments='{}', name='t